# Inferencer Study *(FUSION)*

In [ ]:
import sys

# Add the new path
new_path2 = "/home/michele/code/michele_mmdet3d/"
if not new_path2 in sys.path:
    sys.path.insert(1, new_path2)

# Print the version of MMCV
import mmcv
print(mmcv.__version__)
print(mmcv.__file__)

In [ ]:
#################################################################################################################
#                                           Initialize Inferencer                                               #
#################################################################################################################



# Import the inferencer
from mmdet3d.apis import MultiModalityDet3DInferencer



# Initialize the inferencer class

################################### MVX-NET SMALL VOXEL | STD CONVOLUTION ###################################
# inferencer = MultiModalityDet3DInferencer(model="",
#                                           weights="",
#                                           show_progress=False)

################################### MVX-NET SMALL VOXEL | DEEP CONVOLUTION  ###################################
# inferencer = MultiModalityDet3DInferencer(model="/home/michele/code/michele_mmdet3d/data/minerva_polimove/MVX_SmallVoxels_DeepConvolution/MINERVA_mvxnet_MVX_SmallVoxels_DeepConvolution.py",
#                                           weights="/home/michele/code/michele_mmdet3d/data/minerva_polimove/MVX_SmallVoxels_DeepConvolution/epoch_250_MVX_SmallVoxels_DeepConvolution.pth",
#                                           show_progress=False)



# Define a function that prints the results of the inference
def print_results(results):
    import numpy as np
    for element in results: 
        print("\n\n")
        print(element['predictions'][0]['lidar_path'])
        print(f"scores:\t{element['predictions'][0]['scores_3d']}")
        bboxes = np.array(element['predictions'][0]['bboxes_3d']) 
        num_boxes = bboxes.shape[0]
        for i in range(min(num_boxes, 3)):
            print(f"x: {bboxes[i][0]:.0f}\ty: {bboxes[i][1]:.0f}\tz: {bboxes[i][2]:.0f}")

In [ ]:
#################################################################################################################
#                                               Just Inference                                                  #
#################################################################################################################



# Needed for the handling 
import copy

# Read the files in validation list
val_list_txt_file = "/home/michele/code/michele_mmdet3d/data/minerva_polimove/ImageSets/val.txt"
with open(val_list_txt_file, 'r') as file:
    val_file_names = [line.strip() for line in file]

# Choose a smaller set of the validation files
one_every_n = 1
max_number = 1e3
wait_time_default = 2.5
pred_score_thr_default = 0.2
based_inputs = []
for i, name in enumerate(val_file_names):
    if i%one_every_n==0 and len(based_inputs)<max_number:
        based_inputs.append(dict(
            points=("/home/michele/code/michele_mmdet3d/data/minerva_polimove/training/velodyne_reduced/"+name+".bin"),
            img=("/home/michele/code/michele_mmdet3d/data/minerva_polimove/training/image_2/"+name+".png"),
            infos=("/home/michele/code/michele_mmdet3d/data/minerva_polimove/minerva_polimove_infos_val.pkl")
        ))

# Create a deep copy of the inputs
# NOTE:
#   - Needed because the dictionaries are then modified dinamically
#   - Different from the LIDAR version
inputs = copy.deepcopy(based_inputs)

print(f"Total validation point_clouds: {len(val_file_names)}")
print(f"\tOne every n: {one_every_n}")
print(f"\tMax number: {max_number}")
print(f"\tSelected point_clouds: {len(inputs)}")

# Do the inference
results = []
for input in inputs:
    results.append(inferencer(input))

# Print the information about the predictions
print_results(results)


In [ ]:
#################################################################################################################
#                                           Inference and Visualize                                             #
#################################################################################################################



# Set the matplotlib library so that the correct window is visualized
import matplotlib
matplotlib.use('QtAgg')
import matplotlib.pyplot as plt

# Again, need the deep copy
inputs = copy.deepcopy(based_inputs)

# Do the inference with visualization
for i, input in enumerate(inputs):
    print(f"\n____________________________\n{i+1} out of {len(inputs)}")
    inferencer(input, show=True, wait_time=wait_time_default, pred_score_thr=pred_score_thr_default)

# Model analysis

# Model parameters

In [1]:
import sys

# Add the new path
new_path2 = "/home/michele/code/michele_mmdet3d/"
if not new_path2 in sys.path:
    sys.path.insert(1, new_path2)

In [ ]:
from mmengine.config import Config
from mmengine.runner.checkpoint import (_load_checkpoint,
                                        _load_checkpoint_to_model)
from mmengine.registry import MODELS



# NOTE:
#   - Needed whenever some modules of MMDetection3D are used in a Python Notebook
#   - Otherwise it behaves like if the modules are not registered
from mmdet3d.utils import register_all_modules
register_all_modules()



########################### PATH TO THE CONFIG AND CHECKPOINT FILE
# MVX --> Std Convolution
config_file = '/home/michele/code/michele_mmdet3d/data/minerva_polimove/MVX_SmallVoxels_StdConvolution/MINERVA_mvxnet_MVX_SmallVoxels_StdConvolution.py'
checkpoint_file = '/home/michele/code/michele_mmdet3d/data/minerva_polimove/MVX_SmallVoxels_StdConvolution/epoch_180_MVX_SmallVoxels_StdConvolution.pth'

# # MVX --> Deep Convolution
# config_file = '/home/michele/code/michele_mmdet3d/data/minerva_polimove/MVX_SmallVoxels_DeepConvolution/MINERVA_mvxnet_MVX_SmallVoxels_DeepConvolution.py'
# checkpoint_file = '/home/michele/code/michele_mmdet3d/data/minerva_polimove/MVX_SmallVoxels_DeepConvolution/epoch_270_MVX_SmallVoxels_DeepConvolution.pth'



########################### ACTUAL LOADING
# Load the configuration
cfg = Config.fromfile(config_file)
checkpoint = _load_checkpoint(checkpoint_file, map_location='cpu')
model = MODELS.build(cfg.model)

In [ ]:
########################### TOTAL PARAMETERS
# Computation
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# Printing
print("\n\n")
print(f"Total parameters:\t{total_params:,}")
print(f"Trainable parameters:\t{trainable_params:,} ({100*(trainable_params/total_params):.2f}% of the total)")
print("\n\n")



########################### LAYER BY LAYER
# for name, param in model.named_parameters():
#     print(f"Layer: {name} | Size: {param.size()} | Number of parameters: {param.numel()}")
# print("\n\n")